In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Elsevier / Scopus usage with OPTIONAL Institutional Token.
- Works with just API key if you're on campus/VPN with Scopus/SD entitlements.
- Uses insttoken if provided (off-campus/full access).
- Gracefully skips gated calls if entitlement/auth is missing, but still runs ScienceDirect FullDoc.

Set via env (recommended):
  export ELSEVIER_APIKEY="your_api_key"
  export ELSEVIER_INSTTOKEN="your_inst_token"   # optional

Or put them in config.json:
  { "elsevier_apikey": "...", "elsevier_insttoken": "..." }
"""

from __future__ import annotations
import os, sys, json
from typing import Optional

from elsapy.elsclient import ElsClient
from elsapy.elsprofile import ElsAuthor, ElsAffil
from elsapy.elsdoc import FullDoc, AbsDoc
from elsapy.elssearch import ElsSearch

# ---------------------- Config ----------------------
def load_config() -> dict:
    cfg = {
        # prefer env (UPPERCASE), fall back to config.json, then to empty string
        "elsevier_apikey": os.getenv("ELSEVIER_APIKEY", ""),
        "elsevier_insttoken": os.getenv("ELSEVIER_INSTTOKEN", ""),
    }
    cfg_path = os.getenv("ELSEVIER_CONFIG_JSON", "/home/chen/GNPS_SIM/agent/db_form_agent_system/config.json")
    if os.path.exists(cfg_path):
        try:
            with open(cfg_path, "r") as f:
                file_cfg = json.load(f) or {}
            # only fill if still empty so env always wins
            if not cfg["elsevier_apikey"]:
                cfg["elsevier_apikey"] = file_cfg.get("elsevier_apikey", "")
            if not cfg["elsevier_insttoken"]:
                cfg["elsevier_insttoken"] = file_cfg.get("elsevier_insttoken", "")
        except Exception as e:
            print(f"WARNING: Failed to read config json: {e}", file=sys.stderr)
    return cfg

# ---------------------- Helpers ----------------------
def make_client(cfg: dict) -> ElsClient:
    apikey = (cfg.get("elsevier_apikey") or "").strip()
    if not apikey:
        print("ERROR: Missing Elsevier API key. Set ELSEVIER_APIKEY or put it in config.", file=sys.stderr)
        sys.exit(1)

    client = ElsClient(apikey)
    inst = (cfg.get("elsevier_insttoken") or "").strip()
    if inst:
        client.inst_token = inst  # adds X-ELS-Insttoken automatically
    return client

def has_scopus_entitlement(client: ElsClient) -> bool:
    """Probe Scopus (author search) with a permissive view."""
    try:
        probe = ElsSearch('authlast(einstein)', 'author')
        probe.execute(client, get_all=False, view='STANDARD', count=1)
        return True
    except Exception as e:
        msg = str(e).lower()
        return not ("http 401" in msg or "http 403" in msg or "authorization_error" in msg)

def has_sd_search_entitlement(client: ElsClient) -> bool:
    """Probe ScienceDirect *Search* entitlement (distinct from FullDoc retrieval)."""
    try:
        probe = ElsSearch('test', 'sciencedirect')
        probe.execute(client, get_all=False)  # SD search doesn't support view arg in elsapy
        return True
    except Exception as e:
        msg = str(e).lower()
        return not ("http 401" in msg or "http 403" in msg or "authorization_error" in msg)

def run_sciencedirect_fulltext_examples(client: ElsClient) -> None:
    print("\n=== ScienceDirect FullText examples ===")
    # Full-text by PII
    pii_doc = FullDoc(sd_pii='S1674927814000082')
    if pii_doc.read(client):
        print("PII title:", pii_doc.title)
        pii_doc.write()
    else:
        print("Read SD-by-PII failed.")

    # Full-text by DOI
    doi_doc = FullDoc(doi='10.1016/S1525-1578(10)60571-5')
    if doi_doc.read(client):
        print("DOI title:", doi_doc.title)
        doi_doc.write()
    else:
        print("Read SD-by-DOI failed.")

def run_sciencedirect_search_example(client: ElsClient) -> None:
    print("\n=== ScienceDirect Search example ===")
    try:
        sd_srch = ElsSearch('star trek vs star wars', 'sciencedirect')
        sd_srch.execute(client, get_all=False)
        print("ScienceDirect search results:", len(sd_srch.results))
    except Exception as e:
        print(f"ScienceDirect Search not available (skipping): {e}")

def run_scopus_examples(client: ElsClient) -> None:
    print("\n=== Scopus examples ===")
    # Author by Scopus Author ID (profile)
    my_auth = ElsAuthor(uri='https://api.elsevier.com/content/author/author_id/7004367821')
    if my_auth.read(client):
        print("Author full name:", getattr(my_auth, "full_name", "(n/a)"))
        my_auth.write()
    else:
        print("Read author failed (Scopus).")

    # Affiliation by ID
    my_aff = ElsAffil(affil_id='60101411')
    if my_aff.read(client):
        print("Affiliation name:", getattr(my_aff, "name", "(n/a)"))
        my_aff.write()
    else:
        print("Read affiliation failed (Scopus).")

    # Abstract by Scopus ID
    scp_doc = AbsDoc(scp_id='84872135457')
    if scp_doc.read(client):
        print("Abstract title:", getattr(scp_doc, "title", "(n/a)"))
        scp_doc.write()
    else:
        print("Read abstract-by-Scopus-ID failed (Scopus).")

    # Author search
    auth_srch = ElsSearch('authlast(keuskamp)', 'author')
    auth_srch.execute(client, get_all=False, view='STANDARD', count=25)
    print("Author search results:", len(auth_srch.results))

    # Affiliation search
    aff_srch = ElsSearch('affil(amsterdam)', 'affiliation')
    aff_srch.execute(client, get_all=False, view='STANDARD', count=25)
    print("Affiliation search results:", len(aff_srch.results))

    # Scopus document search
    doc_srch = ElsSearch("AFFIL(dartmouth) AND AUTHOR-NAME(lewis) AND PUBYEAR > 2011", "scopus")
    doc_srch.execute(client, get_all=True, view='STANDARD', count=25)
    print("Scopus doc search results:", len(doc_srch.results))

# ---------------------- Main ----------------------
def main() -> None:
    cfg = load_config()
    client = make_client(cfg)

    # Always run SD full-text (separate product path; you already have this working)
    run_sciencedirect_fulltext_examples(client)

    # Probe ScienceDirect *Search* entitlement (distinct from FullDoc)
    print("\n=== Checking ScienceDirect Search entitlement ===")
    if has_sd_search_entitlement(client):
        print("ScienceDirect Search entitlement detected. Running SD search…")
        run_sciencedirect_search_example(client)
    else:
        print("No ScienceDirect Search entitlement (or off-campus without VPN/insttoken). Skipping SD search.")

    # Probe Scopus entitlement (campus IP or insttoken)
    print("\n=== Checking Scopus entitlement ===")
    if has_scopus_entitlement(client):
        print("Scopus entitlement detected. Running Scopus calls…")
        try:
            run_scopus_examples(client)
        except Exception as e:
            print(f"Scopus calls failed unexpectedly: {e}", file=sys.stderr)
    else:
        print("No Scopus entitlement (or off-campus without VPN/insttoken). Skipping Scopus calls.")

    print("\nDone.")

if __name__ == "__main__":
    main()



=== ScienceDirect FullText examples ===
PII title: Establishing the fair allocation of international aviation carbon emission rights
DOI title: Sensitive Sequencing Method for KRAS Mutation Detection by Pyrosequencing 

=== Checking ScienceDirect Search entitlement ===
No ScienceDirect Search entitlement (or off-campus without VPN/insttoken). Skipping SD search.

=== Checking Scopus entitlement ===
No Scopus entitlement (or off-campus without VPN/insttoken). Skipping Scopus calls.

Done.


In [3]:
config['elsevier_apikey']

'47c7be74b054512c67b94dc0721c1e2b'

In [16]:
import requests
import json

# Specify the search term
query = '"metabolomics"'

# Define the API endpoint URL
url = "http://api.semanticscholar.org/graph/v1/paper/search/bulk"

# Define the query parameters
query_params = {
    "query": query,
    "fields": "title,url,publicationTypes,publicationDate,openAccessPdf",
    "year": "2023-"
}

# Directly define the API key (Reminder: Securely handle API keys in production environments)
api_key = "Y62DwbtSTV85FSMA46QXQ6wekZzaM1Ag5FZPGzCc"  # Replace with the actual API key

# Define headers with API key
headers = {"x-api-key": api_key}

# Send the API request
response = requests.get(url, params=query_params, headers=headers).json()

In [17]:
response

{'total': 31945,
 'token': 'PCOKWVSKJJGM4TWNJNI3EUSQJIWVNUSRKBFEYK2JFUBHFI4VBTGNHEQTZWJZFDCMJWJFGTCSFWGVHTKNRUWVFU6SKIGIZDKNKMGVELGNBSGFGLENSVRGWAIDAEKJ2',
 'data': [{'paperId': '0001867026d8f3bf2b9875e6ce4126314414a26b',
   'url': 'https://www.semanticscholar.org/paper/0001867026d8f3bf2b9875e6ce4126314414a26b',
   'title': 'Ultra-performance liquid chromatography-quadrupole time-of-flight mass spectrometry-based metabolomics to clarify the mechanism of color change of saffron floral bio-residues.',
   'openAccessPdf': {'url': '', 'status': None, 'license': None},
   'publicationTypes': ['JournalArticle'],
   'publicationDate': '2023-01-09'},
  {'paperId': '0001a6100dc411e8d0a4d527b2757d5b61f86bc9',
   'url': 'https://www.semanticscholar.org/paper/0001a6100dc411e8d0a4d527b2757d5b61f86bc9',
   'title': 'Characterisation of the phytochemical and bioactivity profiles of raw tea, stale-aroma, and betelnut-aroma type of Liupao tea through GC/LC-MS-based metabolomics.',
   'openAccessPdf': {